# DeepEval 3.8.8 — Local LLM + HuggingFace Embeddings

**실행 순서대로 셀을 차례로 실행하세요.**

## 0. nest_asyncio 패치 (Jupyter 이벤트 루프 충돌 방지)

> ⚠️ 반드시 가장 먼저 실행

In [1]:
# nest_asyncio 없으면: pip install nest_asyncio
import nest_asyncio
nest_asyncio.apply()
print("nest_asyncio 패치 완료")

nest_asyncio 패치 완료


## 1. 소스 탐색 (실제 파라미터 확인용)

> 한 번만 실행하면 됩니다. 이후 에러 발생 시 여기 출력을 참고하세요.

In [2]:
import inspect, deepeval
from deepeval.synthesizer import Synthesizer
from deepeval.synthesizer.chunking.doc_chunker import DocumentChunker

print("deepeval version:", deepeval.__version__)
print("=" * 60)

print("\n[Synthesizer.__init__ 시그니처]")
print(inspect.signature(Synthesizer.__init__))

print("\n[DocumentChunker.__init__ 시그니처]")
print(inspect.signature(DocumentChunker.__init__))

print("\n[generate_goldens_from_docs 시그니처]")
print(inspect.signature(Synthesizer.generate_goldens_from_docs))

deepeval version: 3.8.8

[Synthesizer.__init__ 시그니처]
(self, model: Union[str, deepeval.models.base_model.DeepEvalBaseLLM, NoneType] = None, async_mode: bool = True, max_concurrent: int = 100, filtration_config: Optional[deepeval.synthesizer.config.FiltrationConfig] = None, evolution_config: Optional[deepeval.synthesizer.config.EvolutionConfig] = None, styling_config: Optional[deepeval.synthesizer.config.StylingConfig] = None, conversational_styling_config: Optional[deepeval.synthesizer.config.ConversationalStylingConfig] = None, cost_tracking: bool = False)

[DocumentChunker.__init__ 시그니처]
(self, embedder: deepeval.models.base_model.DeepEvalBaseEmbeddingModel)

[generate_goldens_from_docs 시그니처]
(self, document_paths: List[str], include_expected_output: bool = True, max_goldens_per_context: int = 2, context_construction_config: Optional[deepeval.synthesizer.config.ContextConstructionConfig] = None, _send_data=True) -> List[deepeval.dataset.golden.Golden]


## 2. xlsx → txt 변환

> FAQ 질문-답변 구조를 그대로 보존합니다.

In [3]:
import pandas as pd

XLSX_PATH = "/home/vsc/LLM_TUNE/QA-FineTune/main/data/고양시도서관 FAQ1.xlsx"
TXT_PATH  = "/home/vsc/LLM_TUNE/QA-FineTune/main/data/고양시도서관_FAQ1.txt"

df = pd.read_excel(XLSX_PATH)

# 컬럼 구조 확인
print("컬럼:", df.columns.tolist())
print(f"행 수: {len(df)}")
df.head(3)

컬럼: ['FAQ', 'TITLE', 'DES']
행 수: 110


,FAQ,TITLE,DES
0,1,회원증을 대리발급 할 수 있나요?,"회원증 대리발급은 만14세 미만 아동, 만65세 이상 어르신, 장애인, 임산부만 가..."
1,2,가족회원이 무엇인가요?,"가족회원이란, 고양시 도서관 정회원으로 가입 후 가족회원으로 등록 시 가족명의로 도..."
2,3,핸드폰이 없는 어린이가 홈페이지 회원가입을 하려면 어떻게 해야 하나요?,본인명의 휴대폰 인증이 불가한 만14세 미만 어린이의 경우 아이핀(I-PIN) 본인...


In [5]:
# ↓ 위 셀 출력을 보고 실제 컬럼명으로 수정하세요
# 예) 질문 컬럼: df.columns[0], 답변 컬럼: df.columns[1]
TITLE = df.columns[1]   # 질문 컬럼명 (또는 직접 "질문" 등으로 지정)
DES= df.columns[2]   # 답변 컬럼명

with open(TXT_PATH, "w", encoding="utf-8") as f:
    for _, row in df.iterrows():
        q = str(row[TITLE]).strip()
        a = str(row[DES]).strip()
        if q and a and q != "nan" and a != "nan":
            f.write(f"Q: {q}\nA: {a}\n\n")

# 변환 결과 확인
with open(TXT_PATH, encoding="utf-8") as f:
    content = f.read()

print(f"저장 완료: {TXT_PATH}")
print(f"파일 크기: {len(content)} 글자")
print("-" * 40)
print(content[:400])

저장 완료: /home/vsc/LLM_TUNE/QA-FineTune/main/data/고양시도서관_FAQ1.txt
파일 크기: 20562 글자
----------------------------------------
Q: 회원증을 대리발급 할 수 있나요?
A: 회원증 대리발급은 만14세 미만 아동, 만65세 이상 어르신, 장애인, 임산부만 가능하며 아래 구비서류를 지참하여 대리인이 방문 시 대리발급이 가능합니다.

· 대상 : 만14세 미만 아동, 만65세 이상 어르신, 장애인, 임산부
· 구비서류
  - 공통 : ①위임자 신분증, ②피위임자(대리인) 신분증
  - 장애인 : 장애인 복지카드 또는 장애인 증명서
  - 임신부 : 산모수첩 / 산모 : 주민등록등본(출산 후 12개월까지)
· 방법 : 홈페이지 회원가입 후 위 항목의 해당하는 구비서류를 지참하여 피위임자(대리인)이 도서관 방문

Q: 가족회원이 무엇인가요?
A: 가족회원이란, 고양시 도서관 정회원으로 가입 후 가족회원으로 등록 시 가족명의로 도서대출·도서예약


## 3. Custom LLM 클래스

In [6]:
import asyncio
from openai import OpenAI, AsyncOpenAI
from deepeval.models.base_model import DeepEvalBaseLLM


class LocalExaone(DeepEvalBaseLLM):
    def __init__(self, model_name: str, base_url: str = "http://localhost:8002/v1"):
        self.model_name = model_name
        self.base_url = base_url
        self._client: OpenAI | None = None
        self._async_client: AsyncOpenAI | None = None

    def load_model(self):
        if self._client is None:
            self._client = OpenAI(api_key="EMPTY", base_url=self.base_url)
        return self._client

    def generate(self, prompt: str, **kwargs) -> str:
        client = self.load_model()
        response = client.chat.completions.create(
            model=self.model_name,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.1,
        )
        return response.choices[0].message.content

    async def a_generate(self, prompt: str, **kwargs) -> str:
        if self._async_client is None:
            self._async_client = AsyncOpenAI(api_key="EMPTY", base_url=self.base_url)
        response = await self._async_client.chat.completions.create(
            model=self.model_name,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.1,
        )
        return response.choices[0].message.content

    def get_model_name(self) -> str:
        return self.model_name


print("LocalExaone 정의 완료")

LocalExaone 정의 완료


## 4. Custom Embedding 클래스

In [7]:
from sentence_transformers import SentenceTransformer
from deepeval.models.base_model import DeepEvalBaseEmbeddingModel


class LocalEmbedding(DeepEvalBaseEmbeddingModel):
    def __init__(self, model_name: str):
        self.model_name = model_name
        self._model: SentenceTransformer | None = None

    def load_model(self) -> SentenceTransformer:
        if self._model is None:
            self._model = SentenceTransformer(self.model_name)
        return self._model

    def embed_text(self, text: str) -> list[float]:
        return self.load_model().encode([text], convert_to_numpy=True)[0].tolist()

    def embed_texts(self, texts: list[str]) -> list[list[float]]:
        return self.load_model().encode(texts, convert_to_numpy=True).tolist()

    async def a_embed_text(self, text: str) -> list[float]:
        loop = asyncio.get_event_loop()
        return await loop.run_in_executor(None, self.embed_text, text)

    async def a_embed_texts(self, texts: list[str]) -> list[list[float]]:
        loop = asyncio.get_event_loop()
        return await loop.run_in_executor(None, self.embed_texts, texts)

    def get_model_name(self) -> str:
        return self.model_name


print("LocalEmbedding 정의 완료")

LocalEmbedding 정의 완료


## 5. 인스턴스 생성

In [8]:
local_llm = LocalExaone(
    model_name="/models/Exaone-3.5-32B-Instruct",
    base_url="http://localhost:8002/v1",
)
local_embed = LocalEmbedding(
    model_name="dragonkue/snowflake-arctic-embed-l-v2.0-ko"
)

# LLM 연결 테스트
try:
    test_resp = local_llm.generate("안녕하세요. 한 문장으로 대답하세요.")
    print("LLM 응답:", test_resp[:80])
except Exception as e:
    print("LLM 연결 실패:", e)

# 임베딩 테스트
try:
    vec = local_embed.embed_text("테스트")
    print(f"임베딩 벡터 차원: {len(vec)}")
except Exception as e:
    print("임베딩 실패:", e)

LLM 응답: 안녕하세요, 무엇을 도와드릴까요?


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

임베딩 벡터 차원: 1024


## 6. DocumentChunker 청킹 진단

> Synthesizer 실행 전에 청킹이 실제로 되는지 먼저 확인합니다.

In [9]:
import asyncio
from deepeval.synthesizer.chunking.doc_chunker import DocumentChunker

# DocumentChunker.__init__ 시그니처 재확인
print("DocumentChunker 시그니처:", inspect.signature(DocumentChunker.__init__))

chunker = DocumentChunker(embedder=local_embed)
asyncio.get_event_loop().run_until_complete(
    chunker.a_load_doc(TXT_PATH, encoding="utf-8")
)

print(f"섹션(청크) 수 : {len(chunker.sections)}")
print(f"총 토큰 수   : {chunker.text_token_count}")
if chunker.sections:
    print("\n첫 번째 청크 샘플:")
    print(chunker.sections[0])
else:
    print("\n⚠️  청크가 0개입니다. chunk_size를 줄여야 합니다.")

DocumentChunker 시그니처: (self, embedder: deepeval.models.base_model.DeepEvalBaseEmbeddingModel)
섹션(청크) 수 : 1
총 토큰 수   : 36535

첫 번째 청크 샘플:
page_content='Q: 회원증을 대리발급 할 수 있나요?
A: 회원증 대리발급은 만14세 미만 아동, 만65세 이상 어르신, 장애인, 임산부만 가능하며 아래 구비서류를 지참하여 대리인이 방문 시 대리발급이 가능합니다.

· 대상 : 만14세 미만 아동, 만65세 이상 어르신, 장애인, 임산부
· 구비서류
  - 공통 : ①위임자 신분증, ②피위임자(대리인) 신분증
  - 장애인 : 장애인 복지카드 또는 장애인 증명서
  - 임신부 : 산모수첩 / 산모 : 주민등록등본(출산 후 12개월까지)
· 방법 : 홈페이지 회원가입 후 위 항목의 해당하는 구비서류를 지참하여 피위임자(대리인)이 도서관 방문

Q: 가족회원이 무엇인가요?
A: 가족회원이란, 고양시 도서관 정회원으로 가입 후 가족회원으로 등록 시 가족명의로 도서대출·도서예약·상호대차 동일하게 이용가능한 서비스입니다.
※ 가족명의 이용 시 자료실 안내데스크에 반드시 본인 회원증을 제시하여야 함(가족 회원증으로 대출 불가)
가족회원 등록방법
가족구성원 모두 정회원 가입 후 가족임을 증명할 수 있는 서류(주민등록등본 또는 가족관계증명서) 지참하여 도서관 방문

Q: 핸드폰이 없는 어린이가 홈페이지 회원가입을 하려면 어떻게 해야 하나요?
A: 본인명의 휴대폰 인증이 불가한 만14세 미만 어린이의 경우 아이핀(I-PIN) 본인인증 과정을 거쳐 홈페이지 가입이 가능합니다.

홈페이지 회원가입 - 만14세 미만 회원가입(어린이회원 가입하기) - 보호자 휴대폰 인증 후 가입자(어린이)의 아이핀을 통해 홈페이지에 가입하실 수 있습니다.

회원가입 바로가기
https://www.goyanglib.or.kr/center/program/memberJoinType.do

Q: 회원증은

## 7. Synthesizer 초기화 (파라미터 자동 분기)

In [12]:
# 셀 A: ContextConstructionConfig 구조 확인
import inspect
from deepeval.synthesizer.config import ContextConstructionConfig
print(inspect.signature(ContextConstructionConfig.__init__))

(self, embedder: Union[str, deepeval.models.base_model.DeepEvalBaseEmbeddingModel, NoneType] = None, critic_model: Union[str, deepeval.models.base_model.DeepEvalBaseLLM, NoneType] = None, encoding: Optional[str] = None, max_contexts_per_document: int = 3, min_contexts_per_document: int = 1, max_context_length: int = 3, min_context_length: int = 1, chunk_size: int = 1024, chunk_overlap: int = 0, context_quality_threshold: float = 0.5, context_similarity_threshold: float = 0.0, max_retries: int = 3) -> None


In [10]:
import os, inspect
from deepeval.synthesizer import Synthesizer

# deepeval이 생성자에서 OpenAI key를 검증하는 경우 대비
os.environ.setdefault("OPENAI_API_KEY", "dummy-not-used")

sig_params = list(inspect.signature(Synthesizer.__init__).parameters.keys())

if "embedder" in sig_params:
    synthesizer = Synthesizer(model=local_llm, embedder=local_embed)
    print("초기화 완료 — embedder= 사용")
elif "embedding_model" in sig_params:
    synthesizer = Synthesizer(model=local_llm, embedding_model=local_embed)
    print("초기화 완료 — embedding_model= 사용")
else:
    synthesizer = Synthesizer(model=local_llm)
    print("초기화 완료 — model= 만 사용")

# 내부에 남아있는 OpenAI 임베딩 속성 강제 교체
replaced = []
for attr in vars(synthesizer):
    if "embed" in attr.lower():
        setattr(synthesizer, attr, local_embed)
        replaced.append(attr)
if not replaced:
    synthesizer.embedder = local_embed
    synthesizer.embedding_model = local_embed
    replaced = ["embedder", "embedding_model"]

print("교체된 속성:", replaced)

초기화 완료 — model= 만 사용
교체된 속성: ['embedder', 'embedding_model']


## 8. 골든 데이터셋 생성

In [11]:
OUTPUT_JSON = "exaone_dataset.json"

goldens = synthesizer.generate_goldens_from_docs(
    document_paths=[TXT_PATH],
    max_goldens_per_context=2,
    include_expected_output=True,
    chunk_size=256,    # 기본 1024보다 작게 → 청크 수 증가
    chunk_overlap=0,
)

print(f"생성된 golden 수: {len(goldens)}")

TypeError: Synthesizer.generate_goldens_from_docs() got an unexpected keyword argument 'chunk_size'

## 9. 결과 저장

In [ ]:
import json

def golden_to_dict(g):
    return {
        "input":             getattr(g, "input",             None),
        "expected_output":   getattr(g, "expected_output",   None),
        "context":           getattr(g, "context",           None),
        "retrieval_context": getattr(g, "retrieval_context", None),
    }

data = [golden_to_dict(g) for g in goldens]

with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print(f"저장 완료 → {OUTPUT_JSON}")
if data:
    print("\n샘플:")
    print(json.dumps(data[0], ensure_ascii=False, indent=2))